# quantum-cq: getting started

This notebook walks through the base workflows of the library: state encoding, metrics, Qiskit export, Navigation V1, logical circuits, engine APIs, the unified pipeline, manual targets and Navigation V2.

The cells are intentionally small and avoid remote execution, credentials or provider discovery.

## 1. Import the public facade

Most workflows start from `CQ`. The package version comes from the installed distribution metadata.

In [ ]:
import quantum_cq
from quantum_cq import CQ

print("quantum-cq", quantum_cq.__version__)

## 2. Encode classical data as a quantum circuit

`CQ.state(...)` creates an encoded circuit object. You can inspect metadata and structural metrics before choosing any backend.

In [ ]:
encoded = CQ.state([1, 0, 1], encoding="basis")

print("encoding:", encoded.encoding_name)
print("object type:", type(encoded).__name__)
print("metrics:", CQ.metrics(encoded))

## 3. Export to Qiskit

Qiskit is the required reference engine. `CQ.to_qiskit(...)` is the legacy exporter path and remains separate from the newer multi-engine layer.

In [ ]:
qc = CQ.to_qiskit(encoded)

print("qubits:", qc.num_qubits)
print("depth:", qc.depth())
print("ops:", dict(qc.count_ops()))
print("legacy exporters:", CQ.available_exporters())

## 4. Navigation V1: addressed memory

Navigation V1 is the stable addressed-memory model. Its logical operation is `|a>|b> -> |a>|b xor D[a]>`.

In [ ]:
memory = CQ.memory([3, 5, 7, 9])
nav_v1 = CQ.nav(memory, engine="explicit")

print("navigation:", nav_v1.metadata["navigation_name"])
print("metadata:", nav_v1.metadata)
print("metrics:", CQ.metrics(nav_v1))

## 5. Build a logical circuit without importing Qiskit

`CQ.circuit(...)` creates a SDK-free logical builder. Emission or compilation happens later through an engine.

In [ ]:
bell = CQ.circuit(2, 2, name="bell")
bell.h(0)
bell.cx(0, 1)
bell.measure(0, 0)
bell.measure(1, 1)
bell_ir = bell.build()

print("logical circuit:", bell_ir.name)
print("format: ir")
print("qiskit depth:", CQ.emit(bell_ir, engine="qiskit").depth())

## 6. Use the multi-engine APIs

`emit`, `compile` and `run_engine` are separate steps. Local Qiskit execution requires the optional `quantum-cq[aer]` extra.

In [ ]:
print("engines:", CQ.engines())
print("qiskit capabilities:", CQ.engine_capabilities("qiskit"))

compiled = CQ.compile(bell_ir, engine="qiskit")
print("compiled engine:", compiled.engine)
print("compiled options:", compiled.options)

try:
    result = CQ.run_engine(bell_ir, engine="qiskit", shots=32)
    print("counts:", result.counts)
except Exception as exc:
    print("execution skipped:", type(exc).__name__, exc)

## 7. Use the unified pipeline

Legacy data + encoding calls preserve their old return type. Enriched calls return `PipelineResult`.

In [ ]:
legacy = CQ.pipeline([1, 0, 1], encoding="basis").build()
enriched = CQ.pipeline(equation="|psi> := H[q0] * |0>").transpile()

print("legacy type:", type(legacy).__name__, legacy.encoding_name)
print("pipeline type:", type(enriched).__name__)
print("scenario status:", enriched.scenario_results[0].status)
print("stages:", [stage.stage_id for stage in enriched.scenario_results[0].stage_results])

## 8. Declare a manual target

Manual targets are neutral descriptors. They do not select a backend and do not imply remote execution.

In [ ]:
target = CQ.manual_target(
    target_id="ideal-two-qubit",
    qubits=2,
    operations=("h", "cx", "measure"),
    topology=(("q0", "q1"),),
    target_type="simulator_ideal",
)

print("target id:", target.descriptor.target_id)
print("target type:", target.descriptor.target_type)
print("qubits:", target.architecture.num_qubits)
print("instructions:", [item.name for item in target.architecture.instructions])

## 9. Navigation V2: finite typed structures

Navigation V2 is explicit. Each `CQ.navigation_v2(...)` call builds one concrete structural operation, validates and canonicalizes the heap, creates a finite plan, lowers to the current CircuitIR when the selected lowering is concrete, and preserves V1 as the default path.

In [ ]:
from quantum_cq import StructuralField, StructuralHeap, StructuralNode, StructuralSelector, StructuralType

node_type = StructuralType(
    "Node",
    (
        StructuralField("payload", "uint", bit_width=2, semantic_role="value"),
        StructuralField("link", "reference", nullable=True, semantic_role="next"),
    ),
)
heap = StructuralHeap(
    (node_type,),
    (
        StructuralNode("tail", "Node", {"payload": 2, "link": None}),
        StructuralNode("head", "Node", {"payload": 1, "link": "tail"}),
    ),
    roots=("head",),
)

nav_v2 = CQ.navigation_v2(heap, operation="read", selector=StructuralSelector.value("payload"))

print("version:", nav_v2.navigation_version)
print("operation:", nav_v2.operation.operation)
print("format:", nav_v2.circuit_format)
print("fingerprint:", nav_v2.plan.equivalence_class.equivalence_fingerprint)
print("memory values:", nav_v2.plan.memory_values)
print("lowering backend:", nav_v2.metadata["lowering_backend"])

## 10. Send Navigation V2 through the same pipeline

The V2 result is accepted by the pipeline through an explicit input adapter, not by ad-hoc `.to_ir()` detection.

In [ ]:
pipe_v2 = CQ.pipeline(structural_navigation=nav_v2).transpile()
scenario = pipe_v2.scenario_results[0]

print("status:", scenario.status)
print("adapter:", scenario.artifacts["input_adapter"].adapter_id)
print("v2 stages:", [stage.stage_id for stage in scenario.stage_results if stage.stage_id.startswith("navigation_v2_")])

## What to read next

- `docs/api_overview.md` for public API workflows.
- `docs/navigation_v2_structural_encoding.md` for Navigation V1 versus V2.
- `docs/release_0_2_0.md` for the release summary.
- `docs/engines/capability_matrix.md` for engine capability status.